# Dataset Usage Example

This notebook demonstrates the use of a dataset class defined outside of Hyrax that is used to train 1) the HyraxCNN model
and 2) the local VGG11 model.

We'll download the Galaxy10 dataset from AstroNN to use in this example. It's a collection of ~23k labeled galaxy images from 10 possible classes.

More info here: https://github.com/henrysky/astroNN and here: https://astronn.readthedocs.io/en/stable/galaxy10sdss.html

In [1]:
import pooch

file_path = pooch.retrieve(
    # DOI for Example HSC dataset
    url="http://www.astro.utoronto.ca/~bovy/Galaxy10/Galaxy10.h5",
    known_hash="sha256:969A6B1CEFCC36E09FFFA86FEBD2F699A4AA19B837BA0427F01B0BC6DED458AF",
    fname="Galaxy10.h5",
    path="./data/galaxy_10",
)

In [2]:
from hyrax import Hyrax

h = Hyrax()

We'll specify that we want to use the local Galaxy10 dataset class.

In [3]:
data_request = {
    "train": {
        "data": {
            "dataset_class": "external_hyrax_example.datasets.galaxy10_dataset.Galaxy10Dataset",
            "data_location": "/home/drew/code/external_hyrax_example/docs/pre_executed/data/galaxy_10",
            "fields": ["image", "label"],
            "primary_id_field": "object_id",
            "split_fraction": 0.8,
            # "dataset_config": {
            #     "channels_first": True,
            # },
        },
    },
    "validate": {
        "data": {
            "dataset_class": "external_hyrax_example.datasets.galaxy10_dataset.Galaxy10Dataset",
            "data_location": "/home/drew/code/external_hyrax_example/docs/pre_executed/data/galaxy_10",
            "fields": ["image", "label"],
            "primary_id_field": "object_id",
            "split_fraction": 0.2,
            # "dataset_config": {
            #     "channels_first": True,
            # },
        },
    },
}

h.set_config("data_request", data_request)

# h.set_config("external_hyrax_example.galaxy10_dataset.channels_first", True)

[2026-03-17 17:11:42,576 hyrax.config_utils:INFO] Merging external default config from /home/drew/code/external_hyrax_example/src/external_hyrax_example/default_config.toml
[2026-03-17 17:11:42,583 hyrax.config_utils:WARNING] Runtime config contains key or section 'data_request' which has no default defined. All configuration keys and sections must be defined in /home/drew/code/hyrax/src/hyrax/hyrax_default_config.toml


We'll start by training the HyraxCNN model. This model is from the Hyrax package.

In [4]:
h.set_config("model.name", "HyraxCNN")

[2026-03-17 17:11:42,614 hyrax.config_utils:INFO] Merging external default config from /home/drew/code/external_hyrax_example/src/external_hyrax_example/default_config.toml


We'll need to update the prepare_inputs function that HyraxCNN uses because the dimensions of the Galaxy10 images
are in a different order than what HyraxCNN expects.

In [5]:
@staticmethod
def prepare_inputs(data_dict):
    import numpy as np

    if "data" not in data_dict:
        raise RuntimeError("Unable to find `data` key in data_dict")

    data = data_dict["data"]
    # rotate the dimensions of the numpy array from (batch_size, H, W, C) to (batch_size, C, H, W)
    image = np.asarray(data["image"], dtype=np.float32)
    if image.ndim == 3:  # single data point, convert to (C, H, W)
        image = image.transpose(2, 0, 1)
    else:  # batch of data, convert to (batch_size, C, H, W)
        image = image.transpose(0, 3, 1, 2)

    label = None
    if "label" in data:
        label = np.asarray(data["label"], dtype=np.int64)

    return (image, label)

In [6]:
model = h.model()
model.prepare_inputs = prepare_inputs

Train the model

In [ ]:
model = h.train()

Now we'll update the model that we're training to use the locally defined VGG11 model.
We'll go ahead and update the `prepare_inputs` function for this model as well to appropriately handle
the dimensions of the Galaxy10 data.

In [8]:
h.set_config("model.name", "external_hyrax_example.models.vgg11.VGG11")

[2026-03-17 17:12:32,021 hyrax.config_utils:INFO] Merging external default config from /home/drew/code/external_hyrax_example/src/external_hyrax_example/default_config.toml


In [9]:
model = h.model()
model.prepare_inputs = prepare_inputs

Now we'll train the VGG11 model.

In [ ]:
model = h.train()

Quick loss comparison between the models.

Orange = HyraxCNN
Green = VGG11

![loss_values](../_static/HyraxCNN_vs_VGG11.png)